# Hermes Agent 进化篇：Skills、Hooks、Plugins 与 Cron


## 会调用工具之后，怎样把经验变成持续可用的能力？

第 02 篇介绍了会话、工具、MCP 和持久记忆。接下来还有四个问题：解决过的任务怎样复用，执行过程怎样加入检查，已有工具怎样扩展，以及任务怎样按时间自动运行。

| 本章模块 | 要解决的问题 | 保存或注册的对象 |
| --- | --- | --- |
| Skills | 下次遇到相似任务，怎样少走弯路？ | 工作流程与参考资料 |
| Curator | 自动积累的技能越来越多，怎样维护？ | 使用记录、状态、归档与整理策略 |
| Hooks | 怎样在工具或会话的关键节点执行检查？ | 事件与回调 |
| Plugins | 怎样增加一个新的工具或集成？ | 插件清单、Schema 与处理函数 |
| Cron | 怎样让任务在指定时间或周期运行？ | 调度规则、任务说明与投递目标 |

本篇依据本地《03-进化篇.md》整理，延续前两篇的 Markdown 笔记排版。课件以 v0.16.0 为基准；关键接口另对照了 2026-10-09 读取的官方文档和公开 main 源码，差异在相关小节中注明。

全部单元格均为 Markdown，命令与代码只供阅读。本文没有安装技能或插件、运行窗口示例、创建定时任务或发送消息。


## 先分清：保存事实、保存流程、增加工具和安排执行

这些能力可以一起使用，但不能互相替代。以“每天整理 AI 新闻”为例：

| 对象 | 保存什么 | 例子 |
| --- | --- | --- |
| MEMORY.md / USER.md | 跨会话的稳定事实与偏好 | 用户希望报告用中文、每次保留 5 条 |
| Skill | 怎样完成任务的可复用流程 | 查原始来源、交叉核对、按主题整理 |
| Tool / MCP | 实际可以调用的操作接口 | 搜索网页、读取文件、访问某个 API |
| Hook | 指定事件发生时的处理 | 工具调用前检查参数，完成后记录指标 |
| Plugin | 在代码层接入一组扩展 | 注册新的资讯 API 工具及相关 Hook |
| Cron | 何时启动任务、结果发到哪里 | 每天 09:00 运行新闻整理流程 |

~~~text
Skill 给出方法 → Agent 选择工具 → Hook 在事件节点介入
Plugin 可以提供新工具或 Hook → Cron 在指定时间启动任务
~~~

这里的“进化”主要指把经验写成可读文件、再维护这些文件。它不等于训练模型参数，也不保证每次保存的经验都正确。


## Skills：按需加载的工作指南

Skill 的入口是一份 SKILL.md，可以附带参考资料、模板、脚本和素材。模型读取它之后，获得任务流程、注意事项和验证方法，再使用会话中实际可用的工具完成任务。

Hermes 使用渐进式加载，避免一开始就把所有技能全文放入上下文。

| 层级 | 入口 | 模型主要看到什么 |
| --- | --- | --- |
| 索引 | skills_list | 名称、简短描述、类别等发现信息 |
| 主说明 | skill_view(name) | 对应 SKILL.md 正文与元信息 |
| 支持文件 | skill_view(name, path) | 当前任务需要的某份参考资料 |

### 文件存在，不代表所有依赖都已具备

安装了网页搜索 Skill，不一定配置了搜索服务；加载了部署 Skill，也不代表已有服务器凭证。技能告诉 Agent“怎样做”，实际能否完成还取决于工具、环境、权限和外部服务。

课件提到 166 个已追踪技能、87 个内置技能、79 个可选技能及 26+ 类别。这些是课件版本的目录统计，会随发行版和安装状态变化，以当前实例的列表为准。


## 查看、搜索、安装与调用技能

先在系统终端查看技能和来源，再决定安装对象。

~~~bash
hermes skills list
hermes skills browse
hermes skills search honcho
hermes skills install --help
~~~

课件给出的安装与卸载形式如下，名称和 URL 都需要对应到实际来源：

~~~bash
hermes skills install honcho
hermes skills install https://example.com/my-skill/SKILL.md
hermes skills uninstall honcho
~~~

example.com 是占位地址。当前官方目录更强调带来源的标识，例如 official/...、clawhub/... 或明确的仓库路径；搜索后复制实际安装命令，可以减少同名技能的歧义。

以下是在 Hermes 会话输入框中使用的命令：

~~~text
/skills
/tech-blog 根据这份研究摘要整理一篇中文技术文章
~~~

已安装且满足平台、工具等显示条件的技能可以作为斜杠命令调用。/tech-blog 需要前面已存在对应技能；它不是系统终端中的可执行程序。

agentskills.io 定义的是开放的技能格式标准，不能把它直接等同于一个 Hermes 技能市场。Hermes 的 Skills Hub、目录和 taps 才负责具体来源的发现与安装。


## 技能目录：主说明保持精简，细节放到支持文件

本章把实际 Hermes 数据根目录记作 HERMES_HOME。默认 Linux / macOS / WSL 常见为 ~/.hermes，Windows 原生默认位置延续第 01 篇，为 %LOCALAPPDATA%\hermes；自定义路径或 Profile 应按实际实例定位。

~~~text
<HERMES_HOME>/skills/
├── writing/
│   └── tech-blog/
│       ├── SKILL.md
│       ├── references/
│       │   └── fact-checking.md
│       ├── templates/
│       │   └── article.md
│       ├── scripts/
│       └── assets/
├── .hub/
│   ├── lock.json
│   ├── quarantine/
│   └── audit.log
└── .bundled_manifest
~~~

| 文件或目录 | 用途 |
| --- | --- |
| SKILL.md | 必需的技能入口，说明何时使用、步骤与验证方法 |
| references/ | 按需查阅的知识、排查说明、API 资料 |
| templates/ | 稳定的输出模板 |
| scripts/ | 辅助程序；需要实际运行才能产生效果 |
| assets/ | 图片、数据等配套资源 |
| .hub/ | Hub 来源、安装记录、扫描暂存等状态 |
| .bundled_manifest | 内置技能的同步状态 |

后四类支持目录都不是每个 Skill 必需的。把大量案例直接塞进 SKILL.md，会增加加载成本，也容易让核心流程被细节淹没。


## 具体 Skill 与 umbrella：从一次经验抽出共同步骤

课件把技能粗略分为三个层次：

| 层次 | 课件示例 | 关注范围 |
| --- | --- | --- |
| 具体 Skill | mlops/axolotl | 某个工具或具体任务 |
| 总括型 Skill（umbrella） | mlops/training | 一组相关流程的共同方法 |
| 类别级总括型 Skill | software-development/debugging | 一类问题的诊断与验证方法 |

例如连续排查多个 Python 导入问题之后，可以把“确认解释器 → 检查包路径 → 检查同名文件 → 最小化复现”抽成共同流程，再把不同环境的细节放进 references/。

umbrella 不是把所有旧文件简单拼接成一个长文件。合并时还要保留适用条件、差异和支持文件的相对链接，确认它仍能指导下一次任务。

Skill 是文件内容与组织方式；Bundle 是同时加载若干技能的组合入口。两者解决的问题不同。


## 写一份完整 SKILL.md：技能元信息与文章元信息分开

课件的技术博客示例展示了写作流程，但缺少标准的技能 frontmatter，内部代码段也与外部 Markdown 示例使用了相同围栏。下面补齐技能元信息，并用外层 ~~~、内层 ``` 展示嵌套内容。

这是 writing/tech-blog/SKILL.md 的阅读示例：

~~~markdown
---
name: tech-blog
description: 根据资料撰写并核对中文技术文章
version: "1.0.0"
metadata:
  hermes:
    category: writing
    tags: [writing, fact-checking]
---

# 中文技术博客写作

## When to Use
用户提供技术资料，希望整理成有解释、有示例的中文文章时使用。

## Procedure
1. 先读用户提供的资料，列出 3–5 个值得展开的结论。
2. 核对命令与版本。
3. 按“问题 → 背景 → 原理 → 示例 → 验证”组织正文。
4. 正文以中文为主，首次出现的重要术语保留英文名称。
5. 代码必须放入带语言标识的 Markdown 代码段。
6. 将事实、推断和待确认信息区分清楚，再检查全文。

## Pitfalls
- 不要把来源没有证明的推测写成事实。
- 不要用缺少依赖与输入条件的代码冒充完整示例。
- 不要把某个版本的默认值写成永久规则。

## Verification
- 检查来源、字段名、版本和文件路径。
- 检查代码段是否闭合，表格是否能正常渲染。
- 检查文章是否说明了示例的输入、输出与限制。

## Output Template
文章需要 frontmatter 时，使用下面的文章模板：

```yaml
---
title: "文章标题"
date: "YYYY-MM-DD"
tags: [AI, Agent]
author: "AI+Human"
---
```
~~~

最上面的 name、description 用于发现与识别 Skill；正文里的 title、date 则是它将来生成的文章元信息。不能用文章的 frontmatter 代替技能的 frontmatter。


## 外部技能目录：共享文件怎样进入当前实例？

团队已有共享技能仓库时，可以在 config.yaml 中配置扫描目录：

~~~yaml
skills:
  external_dirs:
    - ~/.agents/skills
    - /home/shared/team-skills
    - ${SKILLS_REPO}/skills
~~~

路径支持 ~ 和 ${VAR} 展开。这里展示的是 Linux / WSL 路径，应替换为所在环境可访问的路径；SKILLS_REPO 也必须已定义。不存在的目录会被跳过。

外部技能会进入技能索引、skills_list、skill_view 和斜杠命令发现，不是复制到本地之后才可用。

### 版本差异：external_dirs 不等于只读

课件把外部目录描述为只读、由 skill_manage 拒绝修改。当前官方文档明确改为“已有技能就地更新”：目录可写时，patch、write_file、remove_file、delete 等操作可以修改外部技能。共享文件需要只读时，应依靠文件权限和实例配置。

当前文档给出的同名优先级是：

~~~text
project → profile-local skills → skills.create_dir → external_dirs
~~~

课件主要讨论“本地同名技能优先于外部技能”。当前版本还加入项目技能和 create_dir；同一目录里两个不同技能重名时，应使用明确相对路径，不要依赖猜测。

Curator 的自动维护范围，与 skill_manage 能否编辑文件是两套规则。外部技能通常不进入 Curator 的自动归档范围，这并不意味着其内容绝不会被其他写入途径修改。


## Skill Bundle：把经常一起用的技能变成一个入口

如果每次写文章都要先核对事实、再按模板写作，可以把现有技能组合起来。

课件使用 hermes skills bundle create ... --skills ...。当前官方文档的入口是独立的 hermes bundles，重复使用 --skill：

~~~bash
hermes bundles create writing-day \
  --skill tech-blog \
  --skill markdown-style \
  -d "中文技术文章与 Markdown 排版"
hermes bundles list
hermes bundles show writing-day
~~~

tech-blog、markdown-style 必须是当前实例实际可解析的技能，示例不会自动安装它们。组合文件位于数据根目录的 skill-bundles/：

~~~yaml
name: writing-day
description: 中文技术文章与 Markdown 排版
skills:
  - tech-blog
  - markdown-style
instruction: |
  先核对资料中的事实、命令和版本，再整理正文。
  最后检查标题层级、表格与代码段。
~~~

会话内调用：

~~~text
/writing-day 根据这份课程资料整理学习笔记
~~~

Bundle 一次载入多个工作指南，不是在后台逐个运行多个独立 Agent。当前文档指出缺失技能会被跳过并提示；因此“组合能够加载”还要结合实际加载清单检查。


## 条件激活与平台过滤：有这个文件，也可能暂时看不到

技能可以根据可用工具和运行平台决定是否显示。下面展示一个需要 terminal、在 web 工具集缺席时显示的 Linux / macOS 技能元信息：

~~~yaml
name: local-search-fallback
description: 在缺少 web 工具时使用本地搜索流程
platforms: [linux, macos]
metadata:
  hermes:
    category: research
    fallback_for_toolsets: [web]
    requires_toolsets: [terminal]
~~~

这是 frontmatter 的内容片段，放入 SKILL.md 时还要用 --- 包围，并补上正文。

| 字段 | 作用 |
| --- | --- |
| fallback_for_toolsets | 指定工具集可用时隐藏该后备技能 |
| fallback_for_tools | 以具体工具为单位设置后备显示条件 |
| requires_toolsets | 指定工具集缺少时隐藏该技能 |
| requires_tools | 以具体工具为单位设置必需条件 |
| platforms | 在不匹配的系统中隐藏索引、列表与斜杠入口 |

条件激活是在选择可见指南，不是在自动修复失效服务。后备流程仍需要对应脚本、网络或依赖。platforms: [linux, macos] 也不会让 Linux 命令自动变成 Windows 命令；WSL、Windows 原生与远程后端要按实际运行环境判断。


## skill_manage：让 Agent 把经验写成文件

skill_manage 是模型可调用的技能管理工具，不是系统终端命令。课件展示顶层 action/name/content 等参数；当前公开 main 向模型提供的是 operations 数组，每项描述一个操作。

| 操作 | 当前主要参数 | 用途 |
| --- | --- | --- |
| create | name、content，category 可选 | 创建完整 SKILL.md |
| patch：局部修改 | name、old_string、new_string | 精确替换，优先用于小改动 |
| patch：整体重写 | name、content | 替换整份 SKILL.md |
| write_file | name、file_path、file_content | 写支持文件 |
| remove_file | name、file_path | 移除支持文件 |
| delete | name | 删除技能，仍受相应保护规则限制 |

当前实现保留旧的扁平参数用于历史兼容，edit 是整体重写的旧别名；新示例按当前公布的 operations Schema 编写。

课件中的 description、umbrella、reason 不应作为顶层参数直接照抄到当前 Schema。description 放在 SKILL.md frontmatter，分类放 category 或技能元信息；修改原因写进任务说明或结果记录。

支持文件的 file_path 应相对于技能目录，例如 references/fact-checking.md；不能用它逃逸到其他绝对路径。


## 创建技能：content 是完整文件，而不是一句标题

下面是 skill_manage 的工具调用参数示例。JSON 中的 \n 表示文件换行，实际工具收到的是包含换行的字符串。

~~~json
{
  "name": "skill_manage",
  "arguments": {
    "operations": [
      {
        "action": "create",
        "name": "tech-blog",
        "category": "writing",
        "content": "---\nname: tech-blog\ndescription: 根据资料撰写并核对中文技术文章\n---\n\n# 中文技术博客写作\n\n## When to Use\n根据技术资料整理文章时使用。\n\n## Procedure\n1. 读取来源资料。\n2. 核对命令与版本。\n3. 按问题、原理、示例、验证组织正文。\n\n## Pitfalls\n不把未经验证的推测写成事实。\n\n## Verification\n检查来源链接、代码围栏与标题层级。\n"
      }
    ]
  }
}
~~~

这里用一份较短的文件说明参数形状；正式维护时可采用前面的完整写作指南。create 面向新技能，如果名称已经存在，应先读取并采用 patch，而不是把创建当作覆盖操作。

新创建技能默认写入当前 Profile 的 skills/；当前版本配置了 skills.create_dir 时，则写入对应创建目录。路径与分类都应在创建结果中确认。


## 修改经验：先读现有内容，再精确替换

假设前面的 Skill 已有“核对命令与版本。”这一句，新增经验应落到这份指南里。

~~~json
{
  "name": "skill_manage",
  "arguments": {
    "operations": [
      {
        "action": "patch",
        "name": "tech-blog",
        "old_string": "核对命令与版本。",
        "new_string": "核对命令、参数名与版本；遇到来源冲突时注明差异。"
      }
    ]
  }
}
~~~

old_string 必须对应当前文件内容。找不到文本时，应重新读取文件并确认定位，不要随意改用整份覆盖。

补充材料可以单独写入支持文件：

~~~json
{
  "name": "skill_manage",
  "arguments": {
    "operations": [
      {
        "action": "write_file",
        "name": "tech-blog",
        "file_path": "references/fact-checking.md",
        "file_content": "# 技术事实核对\n\n1. 优先核对官方文档和实际接口。\n2. 记录读取日期与适用版本。\n3. 区分课件示例、当前默认值和个人配置。\n"
      }
    ]
  }
}
~~~

写完支持文件后，还应在主说明中加入相对链接，告诉 Agent 在什么情况下读取它。只有文件写入成功，不能证明这个经验真的能复用；还需要用新的相似任务检查指导是否有效。


## 自改进循环：保存可复用方法，也保存它的适用边界

~~~text
遇到新问题 → 解决并验证 → 提炼共同步骤
      ↓
创建或修补 Skill → 下一次按需加载 → 发现偏差再修正
~~~

| 值得保存 | 不宜直接作为通用流程保存 |
| --- | --- |
| 解释器与虚拟环境的确认步骤 | 某次偶然成功但未确认原因的命令 |
| 一个 API 的可靠调用顺序 | 包含真实密钥或私人账号的请求 |
| 排查时需要满足的前置条件 | 把某台机器的路径写成所有机器的默认路径 |
| 怎样判断成功、怎样回退 | 把版本特定行为写成永久结论 |

课件强调“解决复杂问题后保存为 Skill”。学习时应进一步检查：哪些经验已有验证，哪些只是一次结果，哪些可以合并到已有技能。

当前版本还提供技能写入审批配置，例如 skills.write_approval。它影响实际实例的写入流程，本文只介绍机制；可复用指南的质量仍依赖来源核对、文件审阅和后续验证。


## Curator：维护自动积累的技能

技能越来越多时，会出现近似重复、长期不用、描述与实现不同步等问题。Curator 根据使用记录和维护策略，让技能经历状态变化：

~~~text
active（活跃） → stale（长期未用） → archived（移入 .archive/）
                                            ↓
                                      restore 恢复到可用目录
~~~

stale 是维护状态；archive 会把包移到 skills/.archive/，与永久删除不同。不能把“自动归档”理解成“永远丢失”。

### 自动维护有范围

课件的筛选条件包括：位于本地 skills/、不属于 bundled 清单、不在 Hub 的 lock.json 中，并且 .usage.json 有 created_by: agent 或 agent_created: true。

当前文档默认仍管理符合条件的 agent-created 技能；用户手写、外部目录与 Hub 安装技能不默认参与这套自动归档。当前版本还可通过 prune_builtins 显式允许归档闲置内置技能，默认关闭。

“由 AI 写出来”不必然等于“Curator 管理”。当前版本把用户明确要求创建的前台技能标记为 learn，它们默认不进入自动维护；背景自改进产生或显式 adopt 的技能才可能进入。应查看 curator status 确认实际范围。


## Curator 的触发条件与参数：课件策略和当前默认值分开看

课件的自动触发条件是：功能开启、未暂停、距上次运行足够久，而且 Agent 已空闲足够久。Curator 通过维护检查触发，不是给每份 Skill 创建一个 Cron 任务。

下面保留课件的 30 / 90 天作为显式配置，并加入当前版本的 consolidate 开关：

~~~yaml
curator:
  enabled: true
  interval_hours: 168
  min_idle_hours: 2
  stale_after_days: 30
  archive_after_days: 90
  consolidate: false
~~~

| 参数 | 意义 | 课件给出的值 | 当前文档默认值 |
| --- | --- | --- | --- |
| interval_hours | 维护检查的运行间隔门槛 | 168 小时 | 168 小时 |
| min_idle_hours | 空闲时间门槛 | 2 小时 | 2 小时 |
| stale_after_days | 长期未用后标记 stale | 30 天 | 14 天 |
| archive_after_days | 长期未用后归档 | 90 天 | 30 天 |
| consolidate | 是否启用 LLM 合并与修补阶段 | 课件未列此开关 | false |
| prune_builtins | 是否允许规则阶段归档闲置内置技能 | 课件排除内置技能 | false |

上面的 YAML 是主动选择的维护策略，不是声称当前默认值仍是 30 / 90。interval_hours 也不是精确到秒的保证，实际运行取决于对应 CLI、Gateway 或服务端维护检查。

当前文档说明首次观察通常先记录时间，等待一个完整间隔后才进行首次自动维护；需要提前查看时可用 dry-run。


## Curator 的两阶段：规则归档与模型整理

| 阶段 | 工作方式 | 是否需要 LLM |
| --- | --- | --- |
| 规则阶段 | 根据闲置时间和保护规则转换状态、归档 | 不需要 |
| 整理阶段 | 检查内容漂移、修补流程、合并重叠技能 | 需要，当前默认关闭 |

课件介绍两个阶段连续工作。当前版本需要 consolidate: true 或手动 --consolidate 才进行模型整理；运行一次 Curator，不代表一定会调用模型。

辅助模型可通过 auxiliary.curator 配置。课件模型标识保留如下：

~~~yaml
auxiliary:
  curator:
    provider: openrouter
    model: google/gemini-3-flash-preview
    timeout: 600
~~~

provider、model 需要对应服务仍然可用且已配置凭证。timeout 的单位是秒，它不表示本次审查必定用时 600 秒，也不代表费用固定。

归档或合并一份带 scripts/、templates/ 的技能时，要维护整个文件包和相对链接。仅把 SKILL.md 文字拼到另一个文件里，可能留下不可用的辅助文件路径。


## 查看、预览、暂停、固定与恢复

先查看状态和预览维护结果：

~~~bash
hermes curator status
hermes curator run --dry-run
~~~

课件中的常用管理入口如下，每行表示一种操作，不需要顺序执行整组命令：

~~~bash
hermes curator run
hermes curator run --background
hermes curator pause
hermes curator resume
hermes curator pin tech-blog
hermes curator unpin tech-blog
hermes curator restore tech-blog
hermes curator rollback --list
hermes curator rollback
~~~

当前版本还提供 run --consolidate、backup、list-archived、adopt 等入口。dry-run 不修改技能库，但启用模型整理的预览仍可能涉及模型调用；不能把 dry-run 普遍理解为“零费用”。

| 操作 | 作用 |
| --- | --- |
| pause / resume | 暂停或恢复 Curator 的自动维护 |
| pin / unpin | 调整单份合格技能的删除与归档保护 |
| restore | 恢复一份已归档技能 |
| rollback | 按可用快照或版本支持的记录恢复修改 |

会话内可用 /curator 进入相应管理入口。示例名称只适用于该技能存在、且满足相应操作资格的实例。

### pinned 保护删除，不冻结所有内容

固定技能会跳过自动 stale / archive，模型整理也应避开它；skill_manage delete 会被拒绝。但 patch / edit 仍可进行。pin 不是操作系统只读权限。

当前规则也保护 Cron 引用的技能不被自动归档，包括暂停任务的引用；想同时阻止 skill_manage delete，应按资格使用显式 pin。


## Hooks：在事件发生时插入自己的处理

Hook 由事件触发，例如“工具即将执行”或“当前轮任务已经结束”。它不需要模型每次主动选中，因此适合检查、记录和上下文补充。

| 系统 | 注册入口 | 运行范围 | 常见用途 |
| --- | --- | --- | --- |
| Shell Hooks | config.yaml 的 hooks | CLI、Gateway，以及当前文档所列的桌面聊天入口 | 外部脚本检查、格式化、注入上下文 |
| Plugin Hooks | register(ctx) 中调用 register_hook | CLI / Gateway 等 Agent 路径 | Python 拦截与指标采集 |
| Gateway Hooks | hooks/ 下的 HOOK.yaml + handler.py | 仅 Gateway | 消息平台事件记录、生命周期通知 |

课件主要介绍这三类；当前文档还增加了 Outbound Webhooks，把生命周期事件推送到配置的 HTTP 接收端。学习本章先掌握课件的三个入口。

### 并非所有 Hook 都是被动观察

课件开头说“所有钩子非阻塞、不会影响 Agent”，与后面的 block 示例矛盾。应按具体事件判断：pre_tool_call 可以阻止或调整工具调用，pre_llm_call 可以补充上下文，而观察型 Hook 通常忽略返回值。

外部脚本也有实际执行耗时。错误被隔离记录，不等于所有错误都会放行；用于控制的 Hook，其失败策略需要单独确认。


## 常用事件：工具、Agent 轮次与 Gateway 生命周期

| 事件 | 主要时机 | 能做什么 |
| --- | --- | --- |
| pre_tool_call | 工具派发前 | 检查参数；按协议 block / modify |
| post_tool_call | 工具返回后 | 记录耗时和结果；返回值通常被忽略 |
| pre_llm_call | 当前 Agent 轮次进入循环前 | 注入额外上下文 |
| post_llm_call | 成功、未中断的轮次收尾 | 记录最终回复与会话信息 |
| on_session_start | 对应会话初始化 | 初始化状态，具体频率看版本与入口 |
| on_session_end | 当前实现的每轮工作结束 | 清理本轮资源、记录结束状态 |
| gateway:startup | Gateway 启动 | 启动检查或日志 |
| session:start / end / reset | Gateway 会话生命周期 | 记录平台会话变更 |
| agent:start / step / end | Gateway 处理一条消息的过程 | 记录处理阶段与迭代情况 |
| command:* | Gateway 斜杠命令事件 | 命令审计 |

### 两个容易误读的名字

课件把 pre_llm_call / post_llm_call 写成“每次 LLM 调用前后”，当前文档按 Agent 轮次描述它们。需要逐次 API 请求统计时，当前版本另有 pre_api_request / post_api_request 等事件。

课件把 on_session_end 写成退出、重置时触发，当前文档明确它在每轮结束后触发。真正的退出或会话终结还有 on_session_finalize，重置还有 on_session_reset。通知脚本要先选对事件，避免每轮都提示“整个会话已关闭”。


## Shell Hook 的配置：command 是程序与参数

下面注册一个工具执行前的检查脚本，示例适用于 Python 可执行文件在 PATH、脚本位于指定 Windows 路径的实例。

~~~yaml
hooks:
  pre_tool_call:
    - matcher: "^terminal$"
      command: "python C:/hermes-demo/danger_guard.py"
      timeout: 5
      fail_closed: true
~~~

| 字段 | 用途 |
| --- | --- |
| hooks 下的事件名 | 决定何时运行 |
| matcher | 可选正则；针对工具事件筛选工具名称 |
| command | 要启动的外部程序及参数 |
| timeout | 超时秒数；当前默认 60，上限 300 |
| fail_closed | 当前 Shell pre_tool_call 支持，检查失败时阻止调用 |

当前文档指出 command 按参数拆分启动，shell=False。直接把管道、重定向、&& 写进 command，不能假定会按 Shell 语法执行；需要 Shell 时，应明确启动 bash 并传入脚本。

Windows 路径含空格时，使用明确的可执行文件路径并在 command 内正确引用，例如：

~~~yaml
hooks:
  on_session_end:
    - command: '"C:/Program Files/Git/bin/bash.exe" "C:/hermes-demo/turn-end.sh"'
      timeout: 15
~~~

两段是不同示例。合并进同一份 config.yaml 时，应保留一个 hooks 节点并合并事件，避免重复 YAML 键覆盖。实际 Python、Git Bash 和脚本路径都要按安装位置替换。


## Shell Hook 协议：stdin 接收 JSON，stdout 返回决定

当前 Shell Hook 的输入不是课件示例里的 name / parameters，而是 tool_name / tool_input：

~~~json
{
  "hook_event_name": "pre_tool_call",
  "tool_name": "terminal",
  "tool_input": {
    "command": "echo hello"
  },
  "session_id": "sess_example",
  "cwd": "C:/project",
  "extra": {}
}
~~~

非工具事件的 tool_name / tool_input 可以为空，事件特有信息在 extra 中；当前协议还可能附带 profile 等字段。处理脚本应容忍额外字段。

阻止工具时，可以返回当前规范形状：

~~~json
{
  "action": "block",
  "message": "该工具调用不符合本地检查规则"
}
~~~

不做处理时返回空对象：

~~~json
{}
~~~

pre_llm_call 注入上下文的返回示例：

~~~json
{
  "context": "本次检查发现工作目录存在未提交修改，请先核对相关文件。"
}
~~~

stdout 是通信通道，应只输出协议 JSON。调试信息写入 stderr，不能把 print 调试行混在 JSON 前后。Shell 与 Plugin 的输入形式不同：后者通常直接接收 Python 关键字参数。


## 一个可读的拦截脚本：修正课件的字段名

课件通过 Bash 多次解析 JSON 并匹配危险命令。下面改成一个 Python 脚本集中读取，使用当前字段，并确保返回值能被 JSON 解析。

danger_guard.py 的阅读示例：

~~~python
import json
import re
import sys


def block(message):
    return {"action": "block", "message": message}


def inspect_call(payload):
    if not isinstance(payload, dict):
        return block("检查脚本收到的输入不是 JSON 对象")
    if payload.get("tool_name") != "terminal":
        return {}

    params = payload.get("tool_input")
    if not isinstance(params, dict):
        return block("terminal 参数格式不正确")
    command = params.get("command")
    if not isinstance(command, str):
        return block("terminal command 必须是字符串")

    patterns = [
        r"\brm\s+-(?:rf|fr)\s+/(?:\s|$)",
        r"\bdrop\s+table\b",
        r"\bmkfs(?:\.|\s)",
    ]
    if any(re.search(pattern, command, re.IGNORECASE) for pattern in patterns):
        return block("命令命中了本地演示拦截规则")
    return {}


def main():
    try:
        response = inspect_call(json.load(sys.stdin))
    except (ValueError, TypeError, OSError):
        response = block("检查脚本无法解析输入")
    sys.stdout.write(json.dumps(response, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    main()
~~~

搭配前面的 pre_tool_call 配置使用。它只示范字段解析、JSON 输出和几条匹配规则，不是完整的 Shell、SQL 或权限检查器。别名、变量、间接脚本、其他工具等都可能绕开简单字符串匹配。

用于真实控制时，要结合具体工具 Schema、路径或目标范围、最小权限以及可靠的失败策略；不要因为“有一段黑名单”就断言执行绝对安全。


## Hook 超时和出错后，到底继续还是阻止？

| 情况 | 当前 Shell pre_tool_call 默认行为 | 配置 fail_closed: true 后 |
| --- | --- | --- |
| 合法 block JSON | 阻止 | 阻止 |
| 合法空对象 {} | 继续 | 继续 |
| 程序无法启动或超时 | 记录警告并继续 | 阻止 |
| stdout 不是合法 JSON | 记录警告并继续 | 阻止 |
| 普通非零退出且无有效决定 | 通常记录警告并继续 | 阻止 |
| pre_tool_call 退出码 2 | 当前版本按阻止处理 | 阻止 |

fail_closed 是当前文档的行为；老版本是否支持要查看对应帮助和实现。该配置对没有阻止能力的观察事件不会产生同样效果。

Python Plugin Hook 使用另一套回调策略。当前 pre_tool_call 的异常与超时按阻止处理；其他有超时边界的观察事件通常跳过失败回调。不能从一种 Hook 的默认值推断另一种。

Shell Hook 首次使用时，Hermes 通常针对 event / command 组合请求信任确认，具体入口和策略看版本。课件 Windows 配置中的 hooks_auto_accept: true 会改变这个流程，本文示例不默认打开它。

Hook 的目录信任、命令信任和实际检查结果也不是一回事：允许启动脚本，只说明该程序可运行，不证明脚本的判断逻辑可靠。


## Windows 通知示例：提示“本轮完成”，脚本拆开写

课件在 Bash 中嵌入了一大段 PowerShell，还包含另一个用户的绝对路径。学习时把启动器和通知窗口拆成两个文件，更容易检查引用和语法。

以下路径都是示例，不需要在阅读笔记时创建或运行。配置使用当前 on_session_end，每轮结束后触发：

~~~yaml
hooks:
  on_session_end:
    - command: "powershell.exe -NoProfile -WindowStyle Hidden -File C:/hermes-demo/launch-popup.ps1"
      timeout: 15
~~~

launch-popup.ps1 与 popup.ps1 放在同一目录。启动器消费 stdin，在后台以隐藏窗口启动 PowerShell，stdout 只返回空对象：

~~~powershell
$null = [Console]::In.ReadToEnd()
$popupPath = Join-Path $PSScriptRoot 'popup.ps1'
$powershellPath = Join-Path $PSHOME 'powershell.exe'
try {
    Start-Process -FilePath $powershellPath -WindowStyle Hidden -ArgumentList @(
        '-NoProfile', '-STA', '-File', ('"{0}"' -f $popupPath)
    ) -ErrorAction Stop | Out-Null
} catch {
    [Console]::Error.WriteLine('通知窗口启动失败')
}
[Console]::Out.WriteLine('{}')
~~~

启动器返回后，通知窗口单独运行。该通知只说明事件已触发，不证明任务内容成功；要确认任务结果仍应读取实际回复与日志。Windows 执行策略、桌面会话和图形环境也会影响是否能显示。


## 通知窗口正文：三秒后关闭

popup.ps1 的阅读示例：

~~~powershell
Add-Type -AssemblyName System.Windows.Forms
Add-Type -AssemblyName System.Drawing

$form = New-Object System.Windows.Forms.Form
$form.Text = 'Hermes'
$form.Width = 320
$form.Height = 120
$form.StartPosition = 'CenterScreen'
$form.TopMost = $true
$form.ShowInTaskbar = $false

$label = New-Object System.Windows.Forms.Label
$label.Text = '本轮工作已结束，请查看结果'
$label.AutoSize = $true
$label.Location = New-Object System.Drawing.Point(20, 25)
$form.Controls.Add($label)

$timer = New-Object System.Windows.Forms.Timer
$timer.Interval = 3000
$timer.Add_Tick({ $form.Close() })

try {
    $timer.Start()
    [void]$form.ShowDialog()
} finally {
    $timer.Stop()
    $timer.Dispose()
    $form.Dispose()
}
~~~

Form、Label 和 Timer 负责显示与自动关闭。后台运行来自前一段启动器，不能把“后台启动”解释为所有 Hook 天生不会等待。

如果目标是退出整个会话时通知，应先确认所用版本支持的终结事件，例如当前的 on_session_finalize，再调整配置和文案。当前示例没有实际弹出窗口。


## Gateway Hook：用两份文件订阅消息平台事件

Gateway 专用 Hook 放在数据根目录 hooks/，与用于外部脚本的 agent-hooks/ 概念分开。

~~~text
<HERMES_HOME>/hooks/activity-log/
├── HOOK.yaml
└── handler.py
~~~

HOOK.yaml 声明事件：

~~~yaml
name: activity-log
description: 记录 Gateway 的处理阶段
events:
  - gateway:startup
  - agent:start
  - agent:end
  - command:*
~~~

handler.py 示例只记录有限元信息，不发送网络请求：

~~~python
import json
from datetime import datetime, timezone
from pathlib import Path

LOG_FILE = Path(__file__).resolve().parent / "activity.jsonl"


def handle(event_type: str, context: dict):
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event": event_type,
        "platform": context.get("platform"),
        "session_id": context.get("session_id"),
        "command": context.get("command"),
    }
    with LOG_FILE.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(entry, ensure_ascii=False) + "\n")
~~~

处理函数必须名为 handle，接收 event_type 与 context，可以是普通函数或 async def。command:* 匹配命令事件；实际 context 字段取决于订阅的事件。

当前文档说明 Gateway 在启动时发现并导入这些目录，放置文件本身就是信任入口，不走 plugins.enabled。CLI 不会因为存在该目录而运行 Gateway Hook。代码运行身份和访问范围来自 Gateway 进程，应查看处理程序再安装。


## Plugins：把新能力注册进 Hermes

Skill 提供工作方法，Plugin 则可以提供新的代码接口。插件用 register(ctx) 接入，ctx 是 Hermes 提供的注册入口。

| 扩展对象 | 作用 |
| --- | --- |
| Tool | 注册模型可调用的操作，连接 Schema 与 handler |
| Hook | 注册事件回调 |
| Slash / CLI 命令 | 增加会话入口或终端子命令 |
| Skill 与静态资源 | 附带流程、模板与配置资料 |
| Gateway adapter | 扩展消息平台接入 |
| Backend / provider | 扩展记忆、上下文、媒体生成或推理等后端 |
| 会话注入 | 把外部事件或数据送入会话，按版本及权限配置 |

课件提到 v0.14+ 的 ctx.llm，以及自定义模型 ProviderProfile。它们属于具体扩展 API，使用前应对照对应版本的开发文档；调用模型还需要可用的提供商配置，并可能产生费用。

### 最小目录

~~~text
<HERMES_HOME>/plugins/hello-world/
├── plugin.yaml
└── __init__.py
~~~

plugin.yaml 说明插件是谁；__init__.py 中的 register(ctx) 决定它实际注册什么。发现文件、启用插件、加载代码、显示工具是几个不同阶段。


## 最小工具插件：Schema、handler 与 register_tool

先用一个没有外部副作用的问候工具理解插件结构。plugin.yaml：

~~~yaml
name: hello-world
version: "1.0"
description: 提供问候工具并记录工具调用名称
~~~

__init__.py：

~~~python
import json
import logging

logger = logging.getLogger(__name__)


def register(ctx):
    schema = {
        "name": "hello_world",
        "description": "Return a greeting for the supplied name.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Name to greet"}
            },
            "required": ["name"],
        },
    }

    def handle_hello(params, **kwargs):
        name = params.get("name")
        if not isinstance(name, str) or not name.strip():
            return json.dumps({"success": False, "error": "name must be non-empty"})
        return json.dumps(
            {"success": True, "greeting": f"Hello, {name}!"},
            ensure_ascii=False,
        )

    def record_tool(tool_name, args=None, result=None, **kwargs):
        logger.info("Tool completed: %s", tool_name)

    ctx.register_tool(
        name="hello_world",
        toolset="hello_world",
        schema=schema,
        handler=handle_hello,
    )
    ctx.register_hook("post_tool_call", record_tool)
~~~

| 部分 | 面向谁 | 说明 |
| --- | --- | --- |
| schema | 模型 | 告诉模型工具名称、用途与参数形状 |
| handler | Python 运行时 | 校验输入并实际产生结果 |
| register_tool | Hermes 注册系统 | 把工具定义与处理函数关联起来 |
| register_hook | Hook 分发系统 | 为事件安装回调 |

当前模型读取 schema.description；register_tool 的可选 description 是注册元信息。只在后者写说明，不能假定会自动补回缺失的 Schema 描述。回调保留 **kwargs，能容忍后续版本增加字段。


## 课件 shake_window：把操作系统能力包装成 Tool

课件用“让 Windows 前台窗口轻微晃动”演示插件，并不是 Hermes 内置窗口 API。主要调用链是：

~~~text
模型调用 shake_window → Python handler → PowerShell 子进程
    → Win32 获取前台窗口和矩形 → MoveWindow → 返回执行状态
~~~

| Win32 接口 | 用途 |
| --- | --- |
| GetForegroundWindow | 获取当前前台窗口句柄 |
| GetWindowRect | 读取原始位置与大小 |
| MoveWindow | 移动窗口并重绘 |

学习这个示例时，重点看操作边界：窗口来自真实桌面，工具调用会改变用户界面；Win32 返回失败要检查，移动后要恢复位置，Python 子进程要有超时。

下面保留该示例的结构，把 PowerShell 正文拆到 shake.ps1，便于阅读。目录为：

~~~text
<HERMES_HOME>/plugins/shake-window/
├── plugin.yaml
├── __init__.py
└── shake.ps1
~~~

~~~yaml
name: shake-window
version: "1.0"
description: 演示将 Windows 前台窗口操作注册成工具
~~~

此示例面向 Windows 原生桌面。WSL、无桌面服务器、权限隔离或窗口状态不同，行为都可能不同。


## shake.ps1：检查句柄，并在 finally 恢复位置

~~~powershell
$ErrorActionPreference = 'Stop'
Add-Type @"
using System;
using System.Runtime.InteropServices;

[StructLayout(LayoutKind.Sequential)]
public struct RECT {
    public int Left, Top, Right, Bottom;
}

public static class WindowDemo {
    [DllImport("user32.dll")]
    public static extern IntPtr GetForegroundWindow();
    [DllImport("user32.dll")]
    public static extern bool GetWindowRect(IntPtr hWnd, out RECT rect);
    [DllImport("user32.dll")]
    public static extern bool MoveWindow(
        IntPtr hWnd, int x, int y, int width, int height, bool repaint);
}
"@

$windowHandle = [WindowDemo]::GetForegroundWindow()
if ($windowHandle -eq [IntPtr]::Zero) {
    throw '没有可用的前台窗口'
}
$rect = New-Object RECT
if (-not [WindowDemo]::GetWindowRect($windowHandle, [ref]$rect)) {
    throw '无法读取窗口位置'
}
$x = $rect.Left
$y = $rect.Top
$width = $rect.Right - $rect.Left
$height = $rect.Bottom - $rect.Top

try {
    for ($index = 0; $index -lt 8; $index++) {
        foreach ($offset in @(-12, 12)) {
            if (-not [WindowDemo]::MoveWindow(
                $windowHandle, $x + $offset, $y, $width, $height, $true)) {
                throw '移动窗口失败'
            }
            Start-Sleep -Milliseconds 45
        }
    }
} finally {
    if (-not [WindowDemo]::MoveWindow($windowHandle, $x, $y, $width, $height, $true)) {
        [Console]::Error.WriteLine('窗口原始位置恢复失败')
        exit 1
    }
}
~~~

finally 能处理程序正常退出或抛错时的恢复。进程被强制结束、桌面断开或窗口已经销毁时，仍不能保证恢复成功，所以也要检查恢复调用的结果。


## shake-window 的 Python 入口：限制等待并返回结构化结果

__init__.py：

~~~python
import json
from pathlib import Path
import shutil
import subprocess


def register(ctx):
    schema = {
        "name": "shake_window",
        "description": "Briefly move the current Windows foreground window.",
        "parameters": {"type": "object", "properties": {}},
    }

    def handle_shake(params, **kwargs):
        powershell = shutil.which("powershell.exe")
        if powershell is None:
            return json.dumps({"ok": False, "error": "powershell.exe not found"})
        script = Path(__file__).resolve().parent / "shake.ps1"
        if not script.is_file():
            return json.dumps({"ok": False, "error": "shake.ps1 not found"})
        try:
            result = subprocess.run(
                [powershell, "-NoProfile", "-WindowStyle", "Hidden", "-File", str(script)],
                text=True,
                encoding="utf-8",
                errors="replace",
                capture_output=True,
                timeout=10,
                check=False,
            )
        except subprocess.TimeoutExpired:
            return json.dumps({"ok": False, "error": "window operation timed out"})
        except OSError as exc:
            return json.dumps({"ok": False, "error": str(exc)}, ensure_ascii=False)
        return json.dumps(
            {"ok": result.returncode == 0, "stderr": result.stderr.strip()},
            ensure_ascii=False,
        )

    ctx.register_tool(
        name="shake_window",
        toolset="desktop_fun",
        schema=schema,
        handler=handle_shake,
    )
~~~

参数以列表传给 subprocess.run，脚本路径不会被拼成一条待解析的 Shell 命令。timeout 防止处理函数无限等待，但超时终止进程时，PowerShell 的 finally 不保证完成。

工具返回的 JSON 要区分 ok 与 error；模型收到字符串结果，也应检查业务状态。本文仅静态整理该示例，没有移动任何窗口。


## 插件的发现、启用与禁用

| 来源 | 位置或方式 | 课件说明 |
| --- | --- | --- |
| Bundled | 仓库内 plugins/ | 随 Hermes 发布的扩展 |
| User | 数据根目录 plugins/ | 用户本地插件 |
| Project | 当前项目 .hermes/plugins/ | 默认不扫描，需显式设置 HERMES_ENABLE_PROJECT_PLUGINS=true |
| Python 包 | hermes_agent.plugins entry points | 通过包的入口点发现 |

项目插件会执行仓库中的 Python 代码，应确认项目可信再开启发现。它与 SKILL.md 的平台过滤不是同一种控制机制。

终端管理入口：

~~~bash
hermes plugins
hermes plugins list
hermes plugins install owner/repo
hermes plugins enable hello-world
hermes plugins disable hello-world
hermes plugins update hello-world
hermes plugins remove hello-world
~~~

owner/repo 是仓库占位标识。对手工放入目录的插件，更新方式也取决于是否有可识别的安装来源，不能把 update 当作任意本地目录都能自动更新。

~~~yaml
plugins:
  enabled:
    - hello-world
  disabled:
    - noisy-plugin
~~~

一般用户工具插件需要显式启用，disabled 优先于 enabled。当前内置平台、后端、模型提供商等基础设施有特殊加载或选择规则，不能把“所有 bundled 插件默认关闭”写成统一规则。

修改后按对应版本重启或新建会话，并检查已加载插件和实际工具列表。安装成功与工具已进入当前会话并不等价。


## Cron：把任务说明交给调度器

Cron 保存“什么时候做、做什么、使用哪些技能、结果投递到哪里”。到期后，调度器为任务启动独立执行环境，普通 Agent 任务在新会话中运行，最后保存并按配置投递结果。

~~~text
定义任务 → 保存 schedule / prompt / skills / deliver
    ↓
调度器检查到期任务 → 独立运行 → 保存输出 → 按目标投递
~~~

课件以 Gateway daemon 为执行入口，约每 60 秒 tick 一次。仅打开一个普通 CLI 聊天窗口，不能据此认定后台调度器一直在线。当前版本可能还有服务端或托管调度入口，应通过 cron status 确认当前实例。

### 新会话需要完整任务说明

“每天把刚才那个结果发给我”依赖当前聊天，定时任务通常无法知道“刚才那个”指什么。应明确数据来源、输出格式、失败处理与目标位置。

课件说 Cron 运行禁用管理工具，以减少递归调度。当前版本默认仍限制定时 Agent 管理其他 Cron，另有显式启用配置；不要把“不可能产生递归任务”当作无条件保证。


## 时间语法：一次性延迟与重复间隔要分清

| 写法 | 当前公开实现的含义 | 适用例子 |
| --- | --- | --- |
| in 30m | 30 分钟后运行一次 | 一次性提醒 |
| 30m | 每 30 分钟运行，默认重复 | 当前版本的裸时长语法 |
| every 2h | 每 2 小时运行 | 周期检查 |
| every day at 9am | 每天 09:00，依版本支持 | 自然语言时间入口 |
| 0 9 * * * | 每天 09:00 | 明确的五字段 Cron |
| 0 9 * * 1-5 | 周一到周五 09:00 | 工作日任务 |
| 2026-10-20T09:00:00+08:00 | 指定绝对时刻运行一次 | 已知日期的一次性任务 |

课件把裸 30m 解释为一次性延迟；当前 tools/cronjob_tools.py 与 cron/jobs.py 明确将裸时长视为周期。新的阅读示例使用 in 30m，避免版本迁移后不断重复提醒。

五字段顺序：

~~~text
分  时  日  月  星期
0   9   *   *   *      → 每天 09:00
0   9   *   *   1-5    → 工作日 09:00
~~~

课件还使用 every 1d at 09:00。自然语言解析形式会随版本调整；要稳定表达“每天上午九点”，本篇采用 0 9 * * *，并核对实际计算出的下一次运行时间。


## 时区与调度精度：09:00 按谁的时钟？

Cron 表达式没有自带时区，必须结合实例配置理解。当前 Hermes 的时区解析优先读取 HERMES_TIMEZONE，再读取 config.yaml 的 timezone，未设置时使用服务器本地时区；多 Profile 服务还会按 Profile 解析。

~~~yaml
timezone: Asia/Shanghai
~~~

| 时间形式 | 要确认的内容 |
| --- | --- |
| 每天 09:00 | Hermes 时区是否为期望地区 |
| in 30m | 从创建时刻算起的延迟 |
| ISO 时间带 +08:00 / Z | 偏移是否表达了目标绝对时刻 |
| 不带偏移的 ISO 时间 | 所用版本如何补入时区 |

当前源码会给没有偏移的时间戳补入配置的 Hermes 时区，老版本可能不同。跨时区部署或迁移时，应同时查看 next_run_at 的日期、时间与偏移。

约 60 秒检查一次，意味着到期任务可能在后续检查时才被发现；任务排队、工具耗时和网络延迟还会增加实际完成时间。Cron 适合定期工作，不应作为毫秒级精确定时机制。

服务器暂停、休眠、重启后如何补跑也与版本和策略有关。调度器在线、任务已到期、任务已开始、任务已完成是不同状态。


## 会话内创建与终端创建：使用两个不同入口

在 Hermes 会话输入框中：

~~~text
/cron add "in 30m" "提醒我检查构建结果"
/cron add "every 2h" "检查指定服务状态并报告异常" --deliver local
/cron list
~~~

在系统终端中：

~~~bash
hermes cron create "in 30m" "提醒我检查构建结果" --deliver local
hermes cron create "every 2h" "检查指定服务状态并报告异常" --deliver local
hermes cron list
hermes cron status
~~~

这些是各自入口的示例，不需要同时执行两组创建命令。第二个 prompt 还需补充具体服务、检查方式和结果格式，才能成为可运行的业务任务。

本文使用 --deliver local 展示“保存本地输出”的方式；它不会给消息平台发送提醒。要接收通知，需要另外配置实际投递目标。

已有技能可使用 --skill 附加，多份技能重复该参数。当前 CLI 的 schedule 与 prompt 是位置参数，不应把 create 的 prompt 写成未经确认的 --prompt 形式。


## 日报示例：让 prompt 在独立会话中也能读懂

以下 CLI 示例明确资料范围、输出格式与失败说明。实际运行还需要网页工具、网络和模型配置。

~~~bash
hermes cron create "0 9 * * *" \
  "整理运行时最近24小时的AI公开新闻。优先原始公告，选5条，每条包含标题、来源链接、发布时间与两句中文说明；没有可靠消息时说明不足，来源无法访问时注明失败原因，不编造内容。" \
  --name ai-morning-brief \
  --deliver local
~~~

对应当前模型工具的参数形状为：

~~~json
{
  "name": "cronjob_manage",
  "arguments": {
    "action": "create",
    "schedule": "0 9 * * *",
    "prompt": "整理运行时最近24小时的AI公开新闻。优先原始公告，选5条，每条包含标题、来源链接、发布时间与两句中文说明；没有可靠消息时说明不足，来源无法访问时注明失败原因，不编造内容。",
    "name": "ai-morning-brief",
    "deliver": "local"
  }
}
~~~

CLI 和工具 JSON 是同一类任务的两种表达，不是两个都要创建。课件称工具为 cronjob；当前 main 对模型公布的名称为 cronjob_manage，内部仍保留 cronjob() 函数。

把最终报告写入最终回复，调度器负责按 deliver 投递。不要在 prompt 中再要求 Agent 额外发送一遍同样消息，否则可能产生重复投递。


## 定时任务管理：先找 ID，再修改具体对象

~~~bash
hermes cron list
hermes cron list --all
hermes cron edit JOB_ID --schedule "every 4h"
hermes cron pause JOB_ID
hermes cron resume JOB_ID
hermes cron run JOB_ID
hermes cron remove JOB_ID
hermes cron status
~~~

JOB_ID 应替换成实际列表中的任务 ID。这里列出的是不同操作入口，不是让一个任务按顺序经历修改、暂停、执行和删除。

| 操作 | 主要意义 |
| --- | --- |
| list / list --all | 查看任务；后者包含更多禁用或完成任务 |
| edit | 修改已有任务，减少重复创建 |
| pause | 暂停后续自动触发 |
| resume | 恢复调度，补跑规则依版本和配置 |
| run | 手动触发；实际执行方式看入口与版本 |
| remove | 移除任务定义，不等于撤销历史执行效果 |
| status | 检查调度器状态与下一次运行 |

会话内还有 /cron pause、/cron resume、/cron run、/cron remove 等对应入口。当前版本支持按任务名称查找，但名称可能重复，歧义时应使用准确 ID。

### run 的版本差异

课件把 run 描述为“下一次 tick 执行”，当前文档部分段落也保留该说法；当前公开实现则已包含立即手动执行路径，模型工具可后台派发，独立 CLI 通常同步等待结果。不要把返回信息笼统解释为“任务已经成功”。

hermes cron tick 是手动驱动到期任务的执行入口，可能一次触发多个任务，它不是纯只读状态查询。


## 结果保存与投递：目标写法对应不同出口

课件的基础存储结构如下，当前版本还可能增加执行历史、日志和其他状态文件：

~~~text
<HERMES_HOME>/cron/
├── jobs.json
└── output/
    └── <job_id>/
        └── <timestamp>.md
~~~

| deliver 写法 | 意义与前提 |
| --- | --- |
| local | 保存本地，不向消息平台投递 |
| origin | 创建任务时记录的来源会话；没有来源时看回退配置 |
| telegram / discord 等 | 投递到该平台配置的默认目标 |
| platform:target | 指定平台与目标，例如 telegram:CHAT_ID |
| all | 当前版本按已连接且配置的 home channel 解析 |
| 多目标逗号组合 | 例如 telegram,discord，需各自目标可用 |
| ntfy 等扩展出口 | 依版本和实际通知配置支持 |

仅写 deliver: telegram 不会自动创建 Bot、获取聊天 ID 或配置凭证。命令终端也不天然对应一个 Telegram 会话；省略 deliver 的来源推断与回退，应查看所用入口的返回说明。

定时任务产出可能被持久化并发送到外部平台，报告不要无意包含凭证或内部资料。本文所有任务都是阅读示例，没有保存真实 jobs.json，也没有配置消息投递。


## 静默任务：正常时不打扰，异常仍要留下结果

课件介绍 [SILENT]：成功运行的最终回复包含该标记时，可抑制投递，本地输出仍保留。对于检查任务，推荐明确要求“没有异常时只回复这一标记”：

~~~text
按已指定的服务地址检查状态。
如果全部健康，最终回复仅为 [SILENT]。
如果检查失败或发现异常，说明时间、对象、异常与证据。
~~~

| 情况 | 需要理解的行为 |
| --- | --- |
| 成功且只有静默标记 | 不投递常规消息，本地仍有记录 |
| 实际运行失败 | 不能靠静默标记把失败改成成功 |
| 正文引用标记 | 可能触发静默逻辑，避免在普通报告中随意引用 |
| 脚本模式输出为空 | 使用另一套空 stdout 的静默机制 |

当前版本还可配置 failure_deliver / --failure-deliver，失败的出口未必与成功相同。是否真的发出故障通知，最终取决于该目标、平台连接与投递策略。

静默是通知选择，不是停止执行。每次调用 LLM 或外部工具仍可能有成本；“没有收到消息”也不能单独证明任务健康，应查看运行记录。


## No-Agent 模式：固定脚本决定结果时，省去 LLM

阈值检查、磁盘剩余空间、固定接口探测等任务，有时无需模型推理。--no-agent 让调度器执行脚本，并把 stdout 当作最终内容。

~~~bash
hermes cron create "every 5m" \
  --no-agent \
  --script disk-watchdog.py \
  --deliver local \
  --name disk-watchdog
~~~

| 模式 | script 的作用 | 是否调用 LLM |
| --- | --- | --- |
| 普通 Agent + script | 先运行脚本，将结果作为任务上下文 | 会 |
| --no-agent + script | 脚本就是任务，stdout 直接作为内容 | 不会 |

脚本需要位于当前数据根目录 scripts/，解析后的路径不能逃出该目录。当前 .sh / .bash 使用 Bash，其他脚本默认通过 Python；Windows 要确认 Bash 在 PATH，不能假定每台机器都有 /bin/bash。

脚本超时可以显式设置：

~~~yaml
cron:
  script_timeout_seconds: 300
~~~

课件说默认 120 秒，当前源码与文档默认已是 3600 秒。上面的 300 是主动选择的上限，仅约束脚本阶段，不代表整个 LLM 任务总时长被限制为五分钟。


## 脚本示例：磁盘正常时不输出，检查失败时非零退出

保存位置是 <HERMES_HOME>/scripts/disk-watchdog.py。以下示例检查脚本所在磁盘，阈值为 1 GiB：

~~~python
from pathlib import Path
import shutil
import sys

MIN_FREE_BYTES = 1024 ** 3


def main():
    root = Path(__file__).resolve().anchor
    try:
        usage = shutil.disk_usage(root)
    except OSError as exc:
        print(f"磁盘检查失败：{exc}", file=sys.stderr)
        return 1

    if usage.free < MIN_FREE_BYTES:
        free_gib = usage.free / (1024 ** 3)
        print(f"磁盘 {root} 剩余空间不足：{free_gib:.2f} GiB")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
~~~

| 脚本结果 | 调度器的主要行为 |
| --- | --- |
| 正常退出、stdout 为空 | 本次静默，不发送常规内容 |
| 正常退出、有 stdout | 按配置投递文本 |
| 非零退出或超时 | 记录失败，并按失败策略处理 |

打印告警文本但返回 0，表示检查本身完成、业务条件发现异常；读取磁盘失败返回 1，则表示脚本执行失败。两者在运行记录中的意义不同。

当前脚本子进程会处理环境变量传递，不能假定自动拿到 Hermes 的全部凭证。脚本需要外部服务变量时，应按所用版本的显式传递规则配置，日志只检查是否存在，不打印秘密值。


## context_from：把上游输出作为下游任务输入

每次 Cron 通常是独立会话，不会自动看见其他任务的输出。context_from 在运行时读取上游已有结果并注入当前 prompt，使“采集 → 筛选 → 整理”能够传递资料。

~~~text
07:00 采集公开资料 → collector 最终输出
07:30 整理与评分   ← 读取 collector 最近完成的输出
08:00 编写简报     ← 读取 ranker 最近完成的输出
~~~

以下是下游任务的当前模型工具参数，UPSTREAM_JOB_ID 必须替换为 list 返回的真实 ID：

~~~json
{
  "name": "cronjob_manage",
  "arguments": {
    "action": "create",
    "name": "ai-news-ranker",
    "schedule": "30 7 * * *",
    "context_from": ["UPSTREAM_JOB_ID"],
    "prompt": "根据注入的上游新闻资料，先确认资料日期与来源，再按相关性、可信度、新颖性选出5条，输出标题、链接、评分和理由。上游资料缺失或过旧时明确报告，不编造数据。",
    "deliver": "local"
  }
}
~~~

当前工具 Schema 的 context_from 是字符串数组，单个上游也写成数组。课件用单字符串或列表调用内部 cronjob()，不要把内部函数的宽松形状直接套到当前模型 Schema。

课件指出该参数主要由工具设置。所核对的当前 CLI create / edit 解析器也没有 --context-from；不要自行编造这个 CLI 参数。

上游信息应放在最终回复中，便于调度器保存并注入。仅写“文件生成了”会使下游收到一条状态说明，而不是完整资料；若必须读文件，要另外明确文件路径与读取步骤。


## 任务链不是依赖调度：错开时间也不保证同一批资料

context_from 读取的是最近已完成的可用输出，不会等待同一 tick 正在执行的上游。07:00 的采集任务若延迟到 07:40 才完成，07:30 的筛选可能拿到昨天的资料。

| 情况 | 可能结果 | 处理思路 |
| --- | --- | --- |
| 上游从未完成 | 没有可注入内容 | 下游明确报告缺失 |
| 上游本轮仍在运行 | 读到上一轮资料 | 检查时间戳或批次号 |
| 上游本轮失败 | 旧结果或失败内容可能进入上下文，依版本实现 | 检查状态、日期和正文 |
| 上游输出过长 | 可能截断 | 将关键结论、链接、日期放前面 |
| 多个上游 | 按引用顺序拼接 | 指明各自来源，避免合并错对象 |

课件给出每份上游输出约 8000 字符的预算；这是实现级参数，不应当作所有版本的永久上限。字符预算也不等同于 Token 预算。

错开 30 分钟可以降低重叠概率，但不能证明输入属于本轮。对强依赖任务，应检查生成时间、数据日期、批次号和完成状态，必要时把全流程放到一个任务里按顺序执行，或使用支持显式依赖的调度机制。

当前版本还提供 continuity，让任务读取自己的前次输出以去重或延续。它与 context_from 读取其他任务不同；也不能据此认为任务继承了完整历史聊天。


## 出现问题时，按“发现 → 加载 → 执行 → 保存 → 投递”排查

| 现象 | 先检查什么 |
| --- | --- |
| Skill 安装后列表里没有 | 数据根目录、Profile、frontmatter、平台与工具条件 |
| /skill-name 加载了另一份 | Bundle 名称冲突、同名优先级、明确相对路径 |
| 修改共享 Skill 后影响别人 | 外部目录是否可写、版本的就地更新行为 |
| Curator 没维护某份技能 | provenance 标记、Hub / bundled 身份、pin、间隔与空闲条件 |
| Curator 没调用模型 | consolidate 是否开启，当前默认是规则维护 |
| Hook 不触发 | 事件名、matcher、加载时机、首次信任状态 |
| Hook 拦截没生效 | tool_name / tool_input 是否正确、stdout JSON、失败策略 |
| Plugin 在目录里但工具不可见 | enabled / disabled、register 是否成功、工具集与当前会话 |
| Cron 列表有任务却没有自动运行 | 调度器在线状态、Profile、时区、next_run_at 与暂停状态 |
| Cron 运行成功但没收到消息 | deliver、平台目标、连接状态、静默标记与投递记录 |
| 下游新闻用的是旧资料 | 上游完成时间、批次、context_from 引用与输出内容 |

优先读取对应版本的帮助、状态和日志，再调整配置。不要一遇到问题就同时修改数据目录、模型、工具集和调度规则，否则难以判断是哪一步产生了影响。

执行结果与通知结果需要分别看：有本地输出，不证明消息已送达；收到通知，也不证明业务数据已经核对正确。


## 本章复习：哪些内容保存，何时运行，谁决定结果？

| 能力 | 最需要记住的一点 |
| --- | --- |
| Skill | 保存可读的流程，按需加载，不能代替工具与依赖 |
| skill_manage | 写入前先读已有内容，局部经验优先 patch |
| Curator | 有管理范围与保护规则；归档和模型整理要分开理解 |
| Hook | 按事件触发，观察、上下文注入、阻止执行不是同一种语义 |
| Plugin | Schema 给模型看，handler 真正执行，启用与加载要确认 |
| Cron | 时间、完整 prompt、运行环境、输出与投递都要明确 |
| No-Agent | 固定脚本无需推理时可以省去 LLM |
| context_from | 传递已有输出，不自动保证上游本轮已完成 |

复习时可以尝试回答：

1. 用户的语言偏好、新闻整理步骤、搜索 API，分别属于记忆、Skill 还是 Tool？
2. 为什么 Skill 已在目录中，还可能不出现在斜杠命令列表里？
3. 一个 Skill 被 pin 后，哪些操作受保护，哪些修改仍能发生？
4. Shell Hook 为什么不能混合输出调试文字与协议 JSON？
5. 当前一次性“30 分钟后提醒”为什么要写 in 30m？
6. 07:30 的任务怎样证明它使用的是今天 07:00 那一批资料？

练习时先手工审阅一份 SKILL.md、一份插件 Schema 和一个自包含 Cron prompt。这样可以在运行前发现字段、来源和时间表达的问题。


## 课件与当前接口的差异速查

| 课件写法或结论 | 本篇采用的处理 |
| --- | --- |
| 技术博客示例缺少技能 frontmatter，嵌套代码围栏相同 | 补 name / description，并区分外层与内层围栏 |
| external_dirs 是只读 | 标明当前可就地写入，按文件权限保证只读 |
| skills bundle create --skills | 当前使用 bundles create 和重复 --skill |
| skill_manage 顶层 action 与若干附加参数 | 新示例使用 operations 数组及各操作公布字段 |
| Curator 的 30 / 90 是默认值 | 保留为显式策略，同时列当前 14 / 30 默认值 |
| Curator 常规运行包含模型整理 | 当前 consolidate 默认 false |
| 所有 Hook 都不影响流程 | 按观察、注入、控制与失败策略说明 |
| Hook 输入 name / parameters | 当前 Shell 使用 tool_name / tool_input |
| on_session_end 是完整会话退出 | 当前每轮结束；终结与重置另有事件 |
| 所有内置或新插件默认关闭 | 一般工具插件需启用，基础设施有例外 |
| 裸 30m 是一次性 | 当前明确使用 in 30m 表达一次性 |
| run 只等待下个 tick | 标明当前立即执行路径与入口差异 |
| Cron 脚本默认超时 120 秒 | 当前默认 3600 秒，示例主动设为 300 |
| 工具名 cronjob，context_from 可是字符串 | 当前模型接口 cronjob_manage，context_from 用数组 |

以上是本次整理时读到的公开接口，不是对用户已安装版本做过运行测验。main 与在线文档仍会更新，复用示例前应对照安装版本的帮助与 Schema。


## 来源与延伸阅读

主要来源是 [本地课程课件压缩包](D:/baidudisk/BaiduNetdiskDownload/1、课件(1)/1.课件.rar) 中的《03-进化篇.md》，不是根据章节名凭空补写。

| 内容 | 官方资料 |
| --- | --- |
| Skills 格式、目录、组合与条件显示 | [Skills System](https://hermes-agent.nousresearch.com/docs/user-guide/features/skills/) |
| Curator 触发、状态、保护与恢复 | [Curator](https://hermes-agent.nousresearch.com/docs/user-guide/features/curator/) |
| Hook 事件、输入输出与失败策略 | [Event Hooks](https://hermes-agent.nousresearch.com/docs/user-guide/features/hooks/) |
| 插件发现、启用与扩展 | [Plugins](https://hermes-agent.nousresearch.com/docs/user-guide/features/plugins/) |
| 插件注册 API | [开发者插件指南](https://hermes-agent.nousresearch.com/docs/developer-guide/plugins/) |
| 定时语法、脚本与任务链 | [Cron](https://hermes-agent.nousresearch.com/docs/user-guide/features/cron/) |

主要接口核对使用公开 main 的 [skill_manager_tool.py](https://github.com/NousResearch/hermes-agent/blob/main/tools/skill_manager_tool.py)、[cronjob_tools.py](https://github.com/NousResearch/hermes-agent/blob/main/tools/cronjob_tools.py)、[plugins.py](https://github.com/NousResearch/hermes-agent/blob/main/hermes_cli/plugins.py)、[cron/jobs.py](https://github.com/NousResearch/hermes-agent/blob/main/cron/jobs.py)、[cron CLI](https://github.com/NousResearch/hermes-agent/blob/main/hermes_cli/cron.py) 与 [时区处理](https://github.com/NousResearch/hermes-agent/blob/main/hermes_time.py)。读取日期：2026-10-09。

补充的最小问候插件、拦截脚本和磁盘检查脚本用于说明课件机制；课件的窗口示例也做了拆分和错误处理。这些示例经过静态语法与结构检查，没有执行课程功能。
